# Estágio 2.1 — Extração de embeddings com VideoMAE

Pega os vídeos salvos pelo Estágio 1 (`videos.csv`) e gera um embedding
espaço-temporal de 768 dimensões por vídeo, usando o
[VideoMAE](https://huggingface.co/MCG-NJU/videomae-base) (`MCG-NJU/videomae-base`).

Diferente do Estágio 1 (que analisa só a thumbnail), aqui o modelo processa
**16 frames reais do vídeo** — é o que dá ao embedding uma noção de
movimento/tempo, não só de imagem estática.

## Como pega os frames sem baixar o vídeo

O `yt-dlp` resolve a URL de streaming direto do vídeo (sem baixar o arquivo
inteiro pro disco), e o OpenCV (`cv2.VideoCapture`) lê os frames direto
desse stream. Isso evita precisar de armazenamento pra vídeos que só
existem no YouTube mesmo.

**16 frames** não é um número arbitrário — é uma exigência do VideoMAE:
o modelo foi pré-treinado especificamente pra receber clipes de 16 frames.
Menos que isso, o embedding não é confiável.


In [ ]:
import cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image
import yt_dlp
from transformers import VideoMAEImageProcessor, VideoMAEModel

video_df = pd.read_csv("../videos.csv")

print("Carregando VideoMAE...")
processor = VideoMAEImageProcessor.from_pretrained("MCG-NJU/videomae-base")
model = VideoMAEModel.from_pretrained("MCG-NJU/videomae-base")
model.eval()

## Extração de frames

Amostra 16 frames uniformemente espaçados ao longo do vídeo. Se o vídeo
tiver menos frames que o necessário (raro, mas acontece com clipes muito
curtos), interpola pra fechar exatos 16.


In [ ]:
def extrair_frames_do_video(video_id, num_frames=16):
    url_youtube = f"https://www.youtube.com/watch?v={video_id}"

    ydl_opts = {
        # Força o filtro a pegar o codec H.264 (avc1) em vez do AV1 (av01) ou VP9 (vp09)
        "format": "bestvideo[vcodec^=avc1][ext=mp4]/bestvideo[ext=mp4]/best",
        "quiet": True,
        "no_warnings": True,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url_youtube, download=False)

        if "url" in info:
            stream_url = info["url"]
        elif "requested_formats" in info:
            stream_url = info["requested_formats"][0]["url"]
        else:
            stream_url = info["formats"][0]["url"]

    cap = cv2.VideoCapture(stream_url)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames <= 0:
        cap.release()
        return []

    passo = total_frames // (num_frames + 1)
    indices = [passo * i for i in range(1, num_frames + 1)]

    images = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            images.append(Image.fromarray(frame_rgb))

    cap.release()

    # Se vierem menos de 16 frames, interpola para fechar exatos 16
    if 0 < len(images) < num_frames:
        idxs = np.linspace(0, len(images) - 1, num_frames).astype(int)
        images = [images[i] for i in idxs]

    return images

## Geração do embedding

Passa os 16 frames pelo VideoMAE e faz *mean pooling* sobre a sequência de
tokens espaço-temporais da última camada — reduz a saída do modelo pra um
único vetor de 768 dimensões representando o vídeo inteiro.

**Nota sobre falhas**: se a extração de frames falhar (vídeo indisponível,
formato não suportado, etc.), a função retorna um vetor de zeros em vez de
lançar uma exceção — isso mantém o lote inteiro rodando sem travar por
causa de 1-2 vídeos problemáticos. O custo é que esses vídeos ficam com um
embedding sem sinal nenhum (ver `../README.md`, seção de limitações).


In [ ]:
def gerar_embedding_videomae(frames):
    if not frames or len(frames) != 16:
        # Retorna vetor zerado de 768d em caso de falha na extração de frames
        return np.zeros(768, dtype=np.float32)

    inputs = processor(images=[frames], return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)
        # Mean pooling espacial/temporal -> [1, 768]
        video_embedding = outputs.last_hidden_state.mean(dim=1)

    # Converte de Tensor PyTorch para Array NumPy unidimensional (768,)
    return video_embedding.squeeze(0).cpu().numpy()

## Rodando pro catálogo inteiro

**Atenção**: esta célula não é resumível — roda por cima de todo o
`video_df` a cada execução. Para um catálogo grande, vale adaptar pra pular
vídeos que já têm embedding salvo (mesmo padrão de `processar_visao.py` no
Estágio 1).


In [ ]:
embeddings_videomae = []

print(f"Processando {len(video_df)} vídeos...")

for i, video_id in enumerate(video_df["id"], 1):
    print(f"[{i}/{len(video_df)}] Processando vídeo: {video_id}...", end=" ")
    try:
        frames = extrair_frames_do_video(video_id, num_frames=16)
        vetor = gerar_embedding_videomae(frames)
        embeddings_videomae.append(vetor)
        print("OK!")
    except Exception as e:
        print(f"Falhou ({e}). Atribuindo vetor zerado.")
        embeddings_videomae.append(np.zeros(768, dtype=np.float32))

video_df["embedding_videomae"] = embeddings_videomae
print("\nProcessamento concluído!")

In [ ]:
video_df.to_csv("../videos_new.csv", index=False)
print(f"Salvo: videos_new.csv ({len(video_df)} vídeos)")